# 2022 Out-of-Sample 테스트
- **훈련**: 2023-09-07 ~ 2023-11-07 (final_input_vol3d.csv)
- **테스트**: 2022-02-24 ~ 2022-03-21 (final_input_vol3d_2022.csv)
- **모델**: HistGradientBoosting (lr=0.01, depth=3, leaf=20, iter=300, l2=0.0)
- **라벨**: ±0.7% 기준 3분류 (down/neutral/up)

In [8]:
import pandas as pd
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score, classification_report

FIN_COLS = [
    'return_1d','return_3d','return_5d','volatility_3d',
    'volume_change','volume_ma_ratio','sector_return_mean',
    'relative_return_to_sector','relative_return_to_market',
    'relative_volatility_to_sector_3d'
]
MKT_COLS = [
    'VIX_Close','VIX_return_1d','VIX_change_3d',
    'SPY_return_1d','SPY_return_3d','QQQ_return_3d',
    'Oil_return_1d','Gold_return_1d','Dollar_return_1d','Treasury10Y_return_1d'
]

def make_labels(market_csv, thr=0.007):
    raw = pd.read_csv(market_csv)
    raw.columns = raw.columns.str.strip()
    raw['Date'] = pd.to_datetime(raw['Date'])
    raw = raw.sort_values(['ticker','Date'])
    raw['next_return'] = raw.groupby('ticker')['Close'].pct_change(1).shift(-1)
    raw['label'] = np.nan
    raw.loc[raw['next_return'] >  thr, 'label'] = 2
    raw.loc[raw['next_return'] < -thr, 'label'] = 0
    raw.loc[(raw['next_return'] >= -thr) & (raw['next_return'] <= thr), 'label'] = 1
    return raw[['Date','ticker','label']]

print('설정 완료')

설정 완료


In [9]:
# ── 2023 훈련 데이터 ─────────────────────────────────────
df23 = pd.read_csv('final_input_vol3d.csv')
df23['Date'] = pd.to_datetime(df23['Date'])
labels23 = make_labels('company_with_market_features.csv')
df23 = df23.merge(labels23, on=['Date','ticker'], how='left')

emo_cols = [c for c in df23.columns if c.startswith('emo_')]
id_cols  = [c for c in df23.columns if c.startswith('sector_') or c.startswith('ticker_')]
FEAT_COLS = id_cols + FIN_COLS + MKT_COLS + emo_cols

train = df23[FEAT_COLS + ['label','Date']].dropna(subset=['label'])
X_tr = train[FEAT_COLS].values
y_tr = train['label'].values.astype(int)

print(f'훈련 피처: {len(FEAT_COLS)}개')
print(f'훈련 샘플: {len(X_tr)}행 ({train["Date"].nunique()} 거래일)')
print(f'클래스: down={sum(y_tr==0)}, neutral={sum(y_tr==1)}, up={sum(y_tr==2)}')

훈련 피처: 158개
훈련 샘플: 5117행 (43 거래일)
클래스: down=1770, neutral=1790, up=1557


In [10]:
# ── 모델 훈련 (전체 2023 데이터) ────────────────────────
model = HistGradientBoostingClassifier(
    learning_rate=0.01, max_depth=3,
    min_samples_leaf=20, max_iter=300,
    l2_regularization=0.0, class_weight='balanced',
    random_state=42
)
model.fit(X_tr, y_tr)
print('훈련 완료')

# 훈련 데이터 자체 성능 (참고용)
tr_pred = model.predict(X_tr)
print(f'Train macro-F1: {f1_score(y_tr, tr_pred, average="macro"):.3f} (참고용, 과적합 가능)')

훈련 완료
Train macro-F1: 0.595 (참고용, 과적합 가능)


In [11]:
# ── 2022 테스트 데이터 ───────────────────────────────────
df22 = pd.read_csv('final_input_vol3d_2022.csv')
df22['Date'] = pd.to_datetime(df22['Date'])
labels22 = make_labels('company_with_market_features_2022.csv')
df22 = df22.merge(labels22, on=['Date','ticker'], how='left')

# 훈련과 동일한 피처 순서로 맞추기
# (2022에 없는 피처 → 0으로 채움, 순서 고정)
for c in FEAT_COLS:
    if c not in df22.columns:
        df22[c] = 0

test = df22[FEAT_COLS + ['label','Date','ticker']].dropna(subset=['label'])
X_te = test[FEAT_COLS].values
y_te = test['label'].values.astype(int)

print(f'테스트 샘플: {len(X_te)}행 ({test["Date"].nunique()} 거래일)')
print(f'클래스: down={sum(y_te==0)}, neutral={sum(y_te==1)}, up={sum(y_te==2)}')

테스트 샘플: 1547행 (13 거래일)
클래스: down=611, neutral=279, up=657


In [12]:
# ── 2022 예측 & 평가 ─────────────────────────────────────
y_pred = model.predict(X_te)

macro_f1 = f1_score(y_te, y_pred, average='macro', zero_division=0)
print(f'2022 Out-of-Sample macro-F1: {macro_f1:.4f}')
print()
print('── 클래스별 성능 ──')
print(classification_report(y_te, y_pred,
                             target_names=['down','neutral','up'],
                             zero_division=0))

2022 Out-of-Sample macro-F1: 0.3698

── 클래스별 성능 ──
              precision    recall  f1-score   support

        down       0.70      0.17      0.28       611
     neutral       0.23      0.40      0.29       279
          up       0.47      0.65      0.54       657

    accuracy                           0.41      1547
   macro avg       0.47      0.41      0.37      1547
weighted avg       0.52      0.41      0.39      1547



In [13]:
# ── 날짜별 macro-F1 추이 ─────────────────────────────────
test = test.copy()
test['pred'] = y_pred
test['true'] = y_te

daily = []
for date, grp in test.groupby('Date'):
    f1 = f1_score(grp['true'], grp['pred'], average='macro', zero_division=0)
    daily.append({'Date': date.date(), 'macro_F1': round(f1,3),
                  'n': len(grp),
                  'down': int((grp['true']==0).sum()),
                  'neutral': int((grp['true']==1).sum()),
                  'up': int((grp['true']==2).sum())})

daily_df = pd.DataFrame(daily)
print('날짜별 macro-F1:')
print(daily_df.to_string(index=False))
print(f'\n평균: {daily_df["macro_F1"].mean():.3f}  std: {daily_df["macro_F1"].std():.3f}')

날짜별 macro-F1:
      Date  macro_F1   n  down  neutral  up
2022-02-24     0.316 119     7       16  96
2022-02-25     0.243 119    38       28  53
2022-02-28     0.224 119    64       16  39
2022-03-01     0.370 119    20       22  77
2022-03-02     0.216 119    64       33  22
2022-03-07     0.296 119    41       19  59
2022-03-10     0.336 119    91       23   5
2022-03-14     0.361 119    22       18  79
2022-03-15     0.222 119    22       24  73
2022-03-16     0.417 119     9       32  78
2022-03-21     0.336 119    34       26  59
2022-04-20     0.185 119    90       17  12
2022-04-21     0.173 119   109        5   5

평균: 0.284  std: 0.078


In [14]:
# ── 섹터별 성능 ──────────────────────────────────────────
print('섹터별 macro-F1:')
sector_map = df22[['ticker','sector']].drop_duplicates().set_index('ticker')['sector']
test['sector'] = test['ticker'].map(sector_map)

for sec, grp in test.groupby('sector'):
    f1 = f1_score(grp['true'], grp['pred'], average='macro', zero_division=0)
    print(f'  {sec:<35} F1={f1:.3f}  (n={len(grp)})')

섹터별 macro-F1:
  Airline_Travel_Logistics            F1=0.337  (n=260)
  Consumer_Discretionary              F1=0.352  (n=260)
  Defense                             F1=0.342  (n=260)
  Energy                              F1=0.472  (n=247)
  Gold_SafeAsset                      F1=0.398  (n=260)
  Technology                          F1=0.283  (n=260)
